In [1]:
conda create -n car_price_ai python=3.10
conda activate car_price_ai
conda install pandas numpy scikit-learn matplotlib seaborn jupyter

SyntaxError: invalid syntax (1577860541.py, line 1)

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# --- ÉTAPE 1 : CRÉATION DU FICHIER DE TEST (Pour éviter l'erreur FileNotFoundError) ---
data = {
    'Year': [2014, 2013, 2017, 2011, 2018, 2015, 2012, 2016, 2019, 2010],
    'Present_Price': [5.59, 9.54, 9.85, 4.15, 12.5, 8.5, 6.2, 10.1, 15.2, 3.5],
    'Selling_Price': [3.35, 4.75, 7.25, 2.85, 10.5, 6.1, 4.2, 8.9, 13.5, 1.8],
    'Kms_Driven': [27000, 43000, 6900, 5200, 15000, 35000, 45000, 22000, 10000, 60000],
    'Fuel_Type': ['Petrol', 'Diesel', 'Petrol', 'Petrol', 'Diesel', 'Petrol', 'Diesel', 'Petrol', 'Diesel', 'Petrol']
}

df_temp = pd.DataFrame(data)
df_temp.to_csv('car_data.csv', index=False)
print("✅ Fichier 'car_data.csv' créé ou mis à jour.")

# --- ÉTAPE 2 : CHARGEMENT ET PRÉPARATION DES DONNÉES ---
df = pd.read_csv('car_data.csv')

# Ingénierie des caractéristiques : Calculer l'âge du véhicule en 2026
df['Age'] = 2026 - df['Year']

# Encodage des variables textuelles (Convertir 'Petrol'/'Diesel' en chiffres)
df = pd.get_dummies(df, columns=['Fuel_Type'], drop_first=True)

# Sélection des colonnes pour l'entraînement (Features et Cible)
X = df.drop(['Selling_Price', 'Year'], axis=1)
y = df['Selling_Price']

# Division des données : 80% entraînement, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ÉTAPE 3 : ENTRAÎNEMENT DU MODÈLE ---
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Évaluation rapide
predictions = model.predict(X_test)
error = mean_absolute_error(y_test, predictions)
print(f"✅ Modèle entraîné. Erreur moyenne : {error:.2f} units.")

# --- ÉTAPE 4 : PRÉDICTION (PASSÉ, PRÉSENT, FUTUR) ---
def predire_prix(prix_neuf, kms, est_diesel, age_voulu):
    # Création d'une ligne de données identique au format d'entraînement
    input_data = pd.DataFrame({
        'Present_Price': [prix_neuf],
        'Kms_Driven': [kms],
        'Age': [age_voulu],
        'Fuel_Type_Petrol': [0 if est_diesel else 1]
    })
    
    # S'assurer que l'ordre des colonnes est le même que X_train
    input_data = input_data[X_train.columns]
    
    prix_estime = model.predict(input_data)[0]
    return prix_estime

# TEST DE SCÉNARIOS
print("\n--- Résultats des tests ---")
print(f"Prix dans le PASSÉ (Age 0, Neuf) : {predire_prix(10, 0, False, 0):.2f}")
print(f"Prix dans le PRÉSENT (Age 5, 50k km) : {predire_prix(10, 50000, False, 5):.2f}")
print(f"Prix dans le FUTUR (Age 10, 100k km) : {predire_prix(10, 100000, False, 10):.2f}")

✅ Fichier 'car_data.csv' créé ou mis à jour.
✅ Modèle entraîné. Erreur moyenne : 2.79 units.

--- Résultats des tests ---
Prix dans le PASSÉ (Age 0, Neuf) : 8.52
Prix dans le PRÉSENT (Age 5, 50k km) : 8.06
Prix dans le FUTUR (Age 10, 100k km) : 7.87


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# --- 1. PRÉPARATION DES DONNÉES (Exemple avec Marques) ---
data = {
    'Marque': ['Toyota', 'Honda', 'BMW', 'Toyota', 'BMW', 'Honda', 'Toyota', 'Renault'],
    'Present_Price': [10.0, 8.0, 25.0, 12.0, 30.0, 9.0, 11.0, 7.0],
    'Selling_Price': [7.0, 5.5, 18.0, 8.5, 22.0, 6.0, 8.0, 4.5],
    'Year': [2021, 2020, 2022, 2021, 2023, 2019, 2022, 2020]
}

df = pd.DataFrame(data)
df['Age'] = 2026 - df['Year']

# Encodage : On transforme les noms de marques en colonnes (0 ou 1)
df_encoded = pd.get_dummies(df, columns=['Marque'])

# Entraînement du modèle
X = df_encoded.drop(['Selling_Price', 'Year'], axis=1)
y = df_encoded['Selling_Price']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# --- 2. FONCTION DE PRÉDICTION INTERACTIVE ---
def estimer_valeur():
    print("--- Simulateur de Prix de Voiture ---")
    marque_saisie = input("Entrez la marque (ex: Toyota, Honda, BMW, Renault) : ")
    prix_actuel = float(input("Entrez le prix actuel du véhicule (en millions) : "))
    
    # Création des données pour le Passé, Présent, Futur
    scenarios = {
        "PASSÉ (Il y a 3 ans)": -3,
        "PRÉSENT (Maintenant)": 0,
        "FUTUR (Dans 3 ans)": 3
    }
    
    print(f"\nRésultats pour une {marque_saisie} :")
    
    for label, decalage in scenarios.items():
        # Préparation d'une ligne de test
        input_row = pd.DataFrame(columns=X.columns)
        input_row.loc[0] = 0 # On initialise tout à 0
        
        # On remplit les valeurs connues
        input_row['Present_Price'] = prix_actuel
        input_row['Age'] = 5 + decalage # On part d'une base d'une voiture de 5 ans
        
        # On active la colonne de la marque si elle existe dans l'entraînement
        col_marque = f"Marque_{marque_saisie}"
        if col_marque in input_row.columns:
            input_row[col_marque] = 1
            
        prediction = model.predict(input_row)[0]
        print(f" > {label} : {prediction:.2f} millions")

# --- 3. LANCER LE TEST ---
estimer_valeur()

--- Simulateur de Prix de Voiture ---


Entrez la marque (ex: Toyota, Honda, BMW, Renault) :  BMW X5 (xDrive40i)
Entrez le prix actuel du véhicule (en millions) :  87 600


ValueError: could not convert string to float: '87 600'

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# 1. Dataset d'entraînement (Marques simples)
data = {
    'Marque': ['BMW', 'Toyota', 'Honda', 'Mercedes', 'BMW', 'Audi', 'Renault'],
    'Present_Price': [55.0, 15.0, 10.0, 60.0, 85.0, 40.0, 7.0], 
    'Selling_Price': [40.0, 11.0, 7.0, 45.0, 65.0, 28.0, 4.5],
    'Age': [3, 2, 5, 3, 2, 4, 5]
}

df = pd.DataFrame(data)
df_encoded = pd.get_dummies(df, columns=['Marque'])
X = df_encoded.drop(['Selling_Price'], axis=1)
y = df_encoded['Selling_Price']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

def estimer_valeur():
    print("--- Simulateur de Prix de Voiture ---")
    
    # Saisie de la marque (on ne garde que le premier mot pour correspondre au dataset)
    marque_complete = input("Entrez la marque et modèle : ")
    marque_principale = marque_complete.split()[0] # Récupère "BMW" si tu tapes "BMW X5"
    
    # Saisie du prix avec nettoyage des espaces
    prix_str = input("Entrez le prix actuel (ex: 87600) : ")
    prix_nettoye = prix_str.replace(" ", "").replace(",", ".") # Enlève les espaces et gère les virgules
    
    try:
        prix_actuel = float(prix_nettoye)
        
        scenarios = {
            "PASSÉ (Il y a 3 ans)": 2, # Âge plus petit
            "PRÉSENT (Maintenant)": 5, # Âge moyen
            "FUTUR (Dans 3 ans)": 8    # Âge plus grand
        }
        
        print(f"\n--- Estimations pour {marque_principale} ---")
        for label, age in scenarios.items():
            input_row = pd.DataFrame(0, index=[0], columns=X.columns)
            input_row['Present_Price'] = prix_actuel
            input_row['Age'] = age
            
            col_name = f"Marque_{marque_principale}"
            if col_name in X.columns:
                input_row[col_name] = 1
            
            prediction = model.predict(input_row)[0]
            print(f" > {label} : {prediction:.2f}")
            
    except ValueError:
        print("Erreur : Veuillez saisir un prix valide sans lettres ni symboles bizarres.")

estimer_valeur()

--- Simulateur de Prix de Voiture ---


Entrez la marque et modèle :  BMW
Entrez le prix actuel (ex: 87600) :  87600



--- Estimations pour BMW ---
 > PASSÉ (Il y a 3 ans) : 56.38
 > PRÉSENT (Maintenant) : 43.81
 > FUTUR (Dans 3 ans) : 43.81


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# --- 1. CHARGEMENT ET CONVERSION ---
# Le dataset utilise des "Lakhs" (100 000 roupies). 
# On utilise un coefficient pour convertir en millions de FCFA.
COEFF_CFA = 750000 

try:
    df = pd.read_csv('car data.csv')
    df.columns = df.columns.str.strip() # Nettoie les noms de colonnes
    print("✅ Dataset chargé avec succès.")
except FileNotFoundError:
    print("❌ Erreur : Placez le fichier 'car data.csv' dans le dossier Anaconda.")

# Conversion des prix en FCFA
df['Selling_Price_CFA'] = df['Selling_Price'] * COEFF_CFA
df['Present_Price_CFA'] = df['Present_Price'] * COEFF_CFA
df['Age'] = 2026 - df['Year']

# --- 2. PRÉPARATION POUR L'IA (Machine Learning) ---
# Encodage des variables texte (Carburant, Transmission, Type de vendeur)
df_ml = pd.get_dummies(df, columns=['Fuel_Type', 'Selling_type', 'Transmission'], drop_first=True)

# Sélection des colonnes d'entraînement
features = ['Present_Price_CFA', 'Driven_kms', 'Age', 'Owner'] + \
           [col for col in df_ml.columns if 'Fuel_Type_' in col or 'Selling_type_' in col or 'Transmission_' in col]

X = df_ml[features]
y = df_ml['Selling_Price_CFA']

# Entraînement du modèle
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --- 3. SIMULATEUR INTERACTIF ---
def simulateur_ia():
    print("\n" + "="*40)
    print("   SIMULATEUR DE PRIX AUTO (FCFA)")
    print("="*40)
    
    try:
        nom = input("Marque et Modèle (ex: BMW X5) : ")
        
        # Nettoyage automatique des espaces dans la saisie du prix
        p_input = input("Prix du modèle NEUF actuel (ex: 60 000 000) : ")
        p_neuf = float(p_input.replace(" ", "").replace("FCFA", ""))
        
        kms_input = input("Kilométrage actuel (ex: 45000) : ")
        kms = int(kms_input.replace(" ", ""))

        print(f"\n--- Analyses pour votre {nom} ---")
        
        # Définition des scénarios temporels
        scenarios = [
            ("VALEUR PASSÉE (Il y a 3 ans)", -3),
            ("VALEUR PRÉSENTE (Maintenant)", 0),
            ("VALEUR FUTURE (Dans 3 ans)", 3)
        ]

        for label, modif in scenarios:
            input_row = pd.DataFrame(0, index=[0], columns=features)
            input_row['Present_Price_CFA'] = p_neuf
            input_row['Age'] = 5 + modif # On simule sur une base d'une voiture de 5 ans
            input_row['Driven_kms'] = kms + (30000 if modif > 0 else 0) # Augmente les kms pour le futur
            
            prediction = model.predict(input_row)[0]
            print(f" > {label} : {prediction:,.0f} FCFA".replace(",", " "))

    except ValueError:
        print("❌ Erreur : Veuillez saisir des nombres valides (sans lettres).")

# Lancer le test
simulateur_ia()

✅ Dataset chargé avec succès.

   SIMULATEUR DE PRIX AUTO (FCFA)


Marque et Modèle (ex: BMW X5) :  BMW X5
Prix du modèle NEUF actuel (ex: 60 000 000) :  60 000 000
Kilométrage actuel (ex: 45000) :  45000



--- Analyses pour votre BMW X5 ---
 > VALEUR PASSÉE (Il y a 3 ans) : 22 061 925 FCFA
 > VALEUR PRÉSENTE (Maintenant) : 22 061 925 FCFA
 > VALEUR FUTURE (Dans 3 ans) : 21 682 950 FCFA
